<p align="left">
  <a href="https://colab.research.google.com/github/wgomezf/CNN_workshop/blob/main/Notebooks/translearn.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200">
  </a>
</p>

# Transfer learning and fine-tuning

The transfer learning technique is used to reuse a LeNet-5 model pre-trained on MNIST for a new object classification task on the Fashion-MNIST dataset.

In [ ]:
#@title Load necessary libraries
import numpy as np                                                    # Numerical array operations
import matplotlib.pyplot as plt                                       # Data plotting/visualization
import tensorflow as tf                                               # Machine learning
from sklearn.model_selection import train_test_split                  # Data split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive

In [ ]:
#@title TO DO: Modify a pre-trained LeNet-5 model for transfer learning

# Load a pre-trained LeNet-5 model from Google Drive
drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/lenet5.keras'
base_model = tf.keras.models.load_model(model_path, compile=False)

# TO DO:
# 1. Extract all the convolutional layers before the flattening layer
# 2. Add a global average pooling (GlobalAveragePooling2D) after the last max-pooling layer
# 3. A fully connected layer with 64 nodes and ReLU activation function
# 4. A fully connected layer with 32 nodes and ReLU activation function
# 5. Add a new softmax layer with 10 output nodes

# --------  Replace these lines by implementing the above instructions ---------
# Create a new networks from the pre-trained LeNet-5 model
inputs = base_model.inputs
outputs = base_model.layers[-2].output # Remove the softmax layer
# Add a new softmax layer
new_output = tf.keras.layers.Dense(units=10, activation='softmax', name='softmax')(outputs)
new_model = tf.keras.models.Model(inputs=inputs, outputs=new_output)
# -------------------------------------------- ---------------------------------

# Freeze trainable layers
unfreeze_all = True #@param {type:"boolean"}
if unfreeze_all:
  new_model.trainable = True # Unfreeze all trainable layers
else:
  fine_tune_at = 5 # Freeze until the last convolutional layer
  for layer in new_model.layers[:fine_tune_at]:
    layer.trainable = False # Freeze

# List of layers of the new model
for i, layer in enumerate(new_model.layers[:]):
  print(f'{i}: {layer.name}, {layer.trainable}')

new_model.summary()

In [ ]:
#@title Load the Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Split data intro training and validation sets
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.15, random_state=42)

# Check feature and targets shapes
print('\nCheck tensor shapes:\n')
print(f'Shape of training images:   {x_train.shape}')
print(f'Shape of training labels:   {y_train.shape}\n')
print(f'Shape of validation images: {x_val.shape}')
print(f'Shape of validation labels: {y_val.shape}\n')
print(f'Shape of test images:       {x_test.shape}')
print(f'Shape of test labels:       {y_test.shape}\n')

In [ ]:
#@title Visualize some training images
# Create a list of class names
classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
# Image visualization
fig, ax = plt.subplots(nrows=10, ncols=10, figsize=(10,10))
fig.subplots_adjust(hspace=1.0, wspace=0.2)
# Loop through each class. Select 10 images of each class to display
print('Training images\n')
for i in range(10):
    # Select images with the current class label
    images = x_train[y_train == i]
    # Loop through each image and display
    for j in range(10):
        # Set the current subplot
        ax[i,j].imshow(images[j], cmap='gray')
        ax[i,j].set_title(classes[i], fontsize=10)
        ax[i,j].axis('off')
# Show the plot and save the plot image
plt.show()

In [ ]:
#@title Image data preparation for CNN learning
# Transform to deep learning tensor
x_train = x_train.reshape(len(x_train), 28, 28, 1).astype('float32')
x_test = x_test.reshape(len(x_test), 28, 28, 1).astype('float32')
x_val = x_val.reshape(len(x_val), 28, 28, 1).astype('float32')

# Image normalization to the range [0,1]
x_train /= 255.0
x_test /= 255.0
x_val /= 255.0

# Convert label encoding to one hot encoding
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)
y_val = tf.keras.utils.to_categorical(y_val, 10)

In [ ]:
#@title Define training setting
# Adam optimizer
adam = tf.keras.optimizers.Adam(learning_rate=0.001)

# Callback for early stopping if validation accuracy does not improve
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
    )

# Compile model
new_model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

In [ ]:
#@title Fine-tune the pre-trained LeNet-5 model
history = new_model.fit(x_train, y_train, # Training set
                        batch_size=128, # Mini-batch size
                        epochs=100,      # Number of epoch
                        validation_data=(x_val, y_val), # Validation set
                        verbose=2,  # Print training results
                        callbacks=[early_stop] # Early stopping
                        )

In [ ]:
#@title Show training and validation graphs
training_acc = history.history['accuracy']
validation_acc = history.history['val_accuracy']
training_loss = history.history['loss']
validation_loss = history.history['val_loss']

epochs = np.arange(len(training_loss))

plt.figure(figsize=(6, 4))
plt.plot(epochs, training_acc, color='blue', label='Training')
plt.plot(epochs, validation_acc, color = 'green', label='Validation')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

print()

plt.figure(figsize = (6, 4))
plt.plot(epochs, training_loss, color='blue', label='Training')
plt.plot(epochs, validation_loss, color = 'green', label='Validation')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Cross-entropy loss')
plt.legend()
plt.show()

In [ ]:
#@title Model prediction on test images
y_pred = new_model.predict(x_test)
ypp = np.argmax(y_pred, axis=1)
ytt = np.argmax(y_test, axis=1)

ACC = 100*np.mean(ypp == ytt)
print(f"Accuracy: {ACC:.2f}%")

cm = confusion_matrix(ytt, ypp)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(colorbar=False, cmap='Blues',xticks_rotation=45)
plt.title('Confusion matrix')
plt.show()

In [ ]:
#@title Image visualization of test images
fig, ax = plt.subplots(nrows=10, ncols=10, figsize=(10,10))
fig.subplots_adjust(hspace=0.5, wspace=0.1)
# Loop through each class. Select 10 images of each class to display
print('Test images\n')
for i in range(10):
    # Select images with the current class label
    idx = ytt == i
    images = x_test[idx]
    y_pred = ypp[idx]
    y_true = ytt[idx]
    # Loop through each image and display
    for j in range(10):
        # Set the current subplot
        ax[i,j].imshow(images[j], cmap='gray')
        ax[i,j].axis('off')
        # Check prediction
        if (y_pred[j] != y_true[j]):
          ax[i,j].set_title(classes[y_pred[j]], color='red', fontsize=8)
        else:
          ax[i,j].set_title(classes[y_pred[j]], color='green', fontsize=8)
# Show the plot and save the plot image
plt.show()